1. `python3.10 -m pip install ipywidgets==7.7.1 jupyterlab==3.6.3 jupyterlab-widgets==3.0.5 notebook==6.5.4 pythreejs==2.4.2 traitlets==5.9.0 widgetsnbextension jupyterlab_pygments==0.1.1  --force-reinstall --user`
1. `sudo apt install nodejs npm`
1. `jupyter lab build`

# Init

In [1]:
#   0
#   1
#   2
#   3
#   4
#   5
#   6
#   7
#   8
#   9
#  10
#  11
#  12
#  13
#  14, 
#  15
#  16
#  17
#  18
#  19
#  20
#  21
#  22
#  23
#  24
#  25
#  26
#  27
#  28
#  29
#  30
#  31
#  32
#  33
#  34
#  35
#  36,
#  37
#  38
#  39
#  40
#  41
#  42
#  43
#  44
#  45, 
#  46
#  47
#  48
#  49
#  50
#  51
#  52
#  53
#  54
#  55
#  56
#  57
#  58
#  59
#  60
#  61
#  62
#  63
#  64
#  65
#  66
#  67
#  68
#  69
#  70, 2026-01-29 - OKAY
#  71
#  72
#  73
#  74
#  75
#  76
#  77
#  78
#  79, 2026-01-29 - OKAY
#  80, 2026-02-02 - OKAY
#  81
#  82
#  83, 
#  84
#  85
#  86
#  87
#  88
#  89
#  90
#  91
#  92
#  93, 2026-02-03 - OKAY
#  94
#  95, 2026-02-03 - OKAY
#  96, 2026-02-03 - OKAY
#  97
#  98
#  99
# 100
# 101
# 102
# 103, 2026-02-03 - MISSING A RED SHOT at ITER 9
# 104
# 105, 2026-02-03 - OKAY
# 106
# 107, 2026-02-03 - OKAY
# 108, 2026-02-03 - OKAY
# 109, 2026-02-03 - OKAY
# 110, 2026-02-03 - OKAY
# 111, 2026-02-03 - White in wrong place
# 112, 2026-02-03 - OKAY
# 113, 2026-02-03 - OKAY
# 114
# 115, 2026-02-03 - OKAY
# 116
# 117
# 118
# 119, 2026-02-03 - OKAY
# 120
# 121
# 122, 2026-02-03 - OKAY
# 123
# 124
# 125, 2026-02-03 - OKAY
# 126,
# 127,
# 128
# 129, 2026-02-03 - OKAY
# 130
# 131 
# 132, 2026-02-03 - OKAY
# 133, 2026-02-03 - OKAY
# 134, 2026-02-03 - OKAY
# 135, 2026-02-03 - OKAY
# 136
# 137
# 138
# 139
# 140
# 141
# 142
# 143
# 144, 2026-02-03 - OKAY
# 145
# 146, 2026-02-03 - OKAY
# 147, 2026-02-04 - OKAY
# 148, 2026-02-04 - OKAY
# 149, 2026-02-04 - OKAY
# 150, 2026-02-04 - OKAY
# 151, 2026-02-04 - OKAY
# 152, 2026-02-04 - OKAY
# 153, 2026-02-04 - OKAY
# 154, 2026-02-04 - OKAY
# 155, 2026-02-04 - OKAY
# 156, 2026-02-04 - OKAY
# 157, 2026-02-04 - OKAY
# 158, 2026-02-04 - OKAY
# 159, 2026-02-04 - OKAY
# 160
# 161, 2026-02-04 - OKAY
_EXP_INDEX = 161

In [2]:
print( f'Experiment Index: {_EXP_INDEX}' )

Experiment Index: 161


In [3]:
########## INIT ####################################################################################
import pickle, os, traceback, json
from collections import deque
from copy import deepcopy
from pprint import pprint
from typing import Deque

import numpy as np
import matplotlib.pyplot as plt

from magpie_control.ur5 import _CAMERA_XFORM

from aspire.env_config import env_var
from aspire.symbols import GraspObj, ObjPose, euclidean_distance_between_symbols, extract_pose_as_homog
from aspire.BlocksTask import set_blocks_env

from TaskPlanner import set_experiment_env
from draw_jupyter import set_render_env
from utils import deep_copy_memory_list

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Open3D INFO] Resetting default logger to print to terminal.


In [4]:
##### Environment && Constants ############################################
set_blocks_env()
set_experiment_env()
set_render_env()

In [5]:
########## SETUP ###################################################################################
_JSON_PATH  = "data/allData.txt"

# _DATA_DRIVE = "DATA_TANK"
_DATA_DRIVE = "STARGAZER/DATA_TANK"

_PLOT_DIR   = "/media/james/FILEPILE/EROM/data/plots/"
# _PLOT_DIR   = "data/plots/"

_RAM_SAVER = True

tests = [
    "KC-KP",
    "SC-KP",
    "KC-SP",
    "SC-SP",
]

longTestNames = [
    "Known Class & Known Pose", 
    "Sensed Class & Known Pose", 
    "Known Class & Sensed Pose", 
    "Sensed Class & Sensed Pose", 
]

# paths = [ f"/media/james/{_DATA_DRIVE}/2025-08_{test}" for test in tests ]
datasets = [
    [ f"/media/james/{_DATA_DRIVE}/2025-08B_{test}" for test in tests ],
    [ f"/media/james/{_DATA_DRIVE}/RWB_2025-09_{test}" for test in tests ],
]

dataLabels = ["RGB", "RBW",]
datNamLong = {
    "RGB": "Red-Green-Blue", 
    "RBW": "Red-Black-White",
}

fNames = [ f"{_PLOT_DIR}{test}" for test in tests  ]

plotExt = ".pdf"

pkls = deque()

for iii, paths in enumerate( datasets ):
    setNam = dataLabels[iii]
    suffix = "_" + setNam
    skip   = False

    for ii, test in enumerate( tests ):
        ##### Init ################################################################
        path     = paths[ii]
        fName    = fNames[ii]
        longTNam = longTestNames[ii]

        ##### Load ################################################################
        try:
            pkls.extend( [os.path.join( path, item ) for item in sorted( os.listdir( path ) ) if ((".pkl" in f"{item}".lower()) and ("_OCV-State" not in f"{item}"))] )
        except FileNotFoundError as e:
            print( f"\n404, SKIP THIS TEST: {e}\n" )
            skip = True
            continue

In [6]:
########## HELPER FUNCTIONS ########################################################################
_TITLE_FONT_SIZE = 13
_N_TRIALS        = 20
_TIGHT_MARGIN    =  0.05

In [7]:
def camPose_from_effPose( effPose : np.ndarray ):
    """ Get the camera pose given the effector pose """
    return np.dot( effPose, _CAMERA_XFORM )


def filter_series( series, stdFactor = 2.0 ):
    """ Filter outliers more than `stdFactor` standard deviations from the mean """
    if not len( series ):
        return list()
    mu = np.mean( series )
    sd = np.std( series )
    nuSeries = deque()
    thresh   = abs(sd*stdFactor)
    for datum in series:
        if abs( datum - mu ) <= thresh:
            nuSeries.append( datum )
    return list( nuSeries )


def as_json( dataObj, asLeaf = False ):
    """ Convert the dataclass into a struct that can be JSON serialized """
    def make_serializable( obj, depth = 0 ):
        nonlocal asLeaf
        if isinstance( obj, (deque, list,) ):
            rtnLst = deque()
            for item in obj:
                rtnLst.append( make_serializable( item, depth+1 ) )
            return list( rtnLst )
        elif isinstance( obj, np.ndarray ):
            return obj.tolist()
        elif isinstance( obj, dict ):
            rtnDct = dict()
            for k, v in obj.items():
                rtnDct[k] = make_serializable( v, depth+1 )
            return rtnDct
        else:
            return obj
    rtnObj = make_serializable( dataObj, 0 )
    return rtnObj


def crash_out():
    """ End the program with Brutal Finality """
    print( "\n\n" )
    os.system( 'kill %d' % os.getpid() ) 


def extract_pose_from_str( poseStr : str ):
    """ Get the homogeneous coordinates from the string and ignore everything else """
    nstLst = list()
    depth  = 0
    numStr = ""
    row    = list()

    def store_num():
        """ Add the number to the row """
        nonlocal row, numStr, poseStr
        if len( numStr ):
            try:
                row.append( float( numStr.strip() ) )
            except ValueError as e:
                print( f"BAD: {e}" )
                print( numStr  )
                print( poseStr )
                crash_out()
        numStr = ""

    def store_row():
        """ Add the row to the array """
        nonlocal nstLst, row
        if len( row ):
            nstLst.append( row )
        row = list()

    for char in poseStr:
        if char == '[':
            depth += 1
        elif char == ']':
            if depth == 2:
                store_num()
            depth -= 1
            if depth == 1:
                store_row()
        elif char == ' ':
            if depth == 2:
                store_num()
        elif char == '\n':
            pass
        elif depth == 2:
            numStr += char
        else:
            pass
            # print( f"`extract_pose_from_str()`, BAD STATE:\n{char}\n{poseStr}\n" )

    try:
        return np.array( nstLst )
    except Exception as e:
        traceback.print_exc()
        print( f"BAD: {e}" )
        crash_out()


def sum_dicts( dct0 : dict, dct1 : dict ):
    """ Return the sum of numeric values """
    rtnDct = dict()
    for dct in [dct0, dct1,]:
        for k, v in dct.items():
            if k in rtnDct:
                tot = 0
                try:
                    tot = v + rtnDct[k]
                except Exception as e:
                    pass
                rtnDct[k] = tot
            else:
                rtnDct[k] = v
    return rtnDct
        

In [8]:
class TColor:
    """ Terminal Colors """
    # Source: https://stackoverflow.com/a/287944
    HEADER    = '\033[95m'
    OKBLUE    = '\033[94m'
    OKCYAN    = '\033[96m'
    OKGREEN   = '\033[92m'
    WARNING   = '\033[93m'
    FAIL      = '\033[91m'
    ENDC      = '\033[0m'
    BOLD      = '\033[1m'
    UNDERLINE = '\033[4m'

# Load File(s)

In [9]:
print( f"Found {len(pkls)} files!" ) 


Found 162 files!


# "Ground Truth" Annotator

In [10]:
from State import OCV_State_Tracker


# Inspect File

In [11]:
from random import choice
from uuid import uuid4

from IPython.display import clear_output

from Memory import Memory
from utils import JupyterPlotServer
from draw_jupyter import render_memory_list
from homog_utils import posn_from_xform

_MEM_IMG_DIR = "data/MemImg"
_SAFE_POSN   = np.array( [-0.25199, -0.26192, 0.47106,] ) 
_VERBOSE     = True # False
_CHK_OUT     = True

for _EXP_INDEX in range( len(pkls) ): 

    print( f'Experiment Index: {_EXP_INDEX} / {len(pkls)-1}' )

    pklFile    = pkls[ _EXP_INDEX ]
    ocvTracker = OCV_State_Tracker( pklFile )

    if "RWB" in f"{pklFile}".upper():
        _BLOCK_NAMES = ['redBlock','blkBlock','whtBlock',]
    else:
        _BLOCK_NAMES = ['redBlock','grnBlock','bluBlock',]
    
    print( f"Looking for {_BLOCK_NAMES}" )
    
    data = list()
    try:
        with open( pklFile, 'rb' ) as f:
            data = pickle.load( f )
            print( f"LOAD SUCCESS: {pklFile}" )
    except EOFError as e:
        print( f"LOAD ERROR: {e}" )

    memory = Memory( suppressRecord = True ) 
    jps    = JupyterPlotServer()
    
    ### Steps ###
    Nstep    = 0
    tStepBgn = 0
    tStepEnd = 0
    tStepDqu = deque()
    
    ### 1. Object Search ###
    shotPoses = list()
    
    ### 2. Symbol Grounding ###
    Nground    = 0
    tGroundBgn = 0
    tGroundEnd = 0
    tGroundDqu = deque()
    symbols_t  = list()
    totFound   = 0
    totConfuse = 0
    NfailFind  = 0
    
    ### 3. Planning ###
    Nplan     = 0
    tPlanBgn  = 0
    tPlanEnd  = 0
    tPlanDqu  = deque()
    NfailPlan = 0
    pNewStep  = False # Has a new step begun?
    pTryPlan  = False # Did we try to plan this step?
    
    ### 4. Acting ###
    Naction    = 0
    tActionBgn = 0
    tActionEnd = 0
    tActionDqu = deque()
    NfailActn  = 0
    actionFail = False
    
    ### N. Analysis ###
    totlConf = {
        "N_total"  : 0,
        "N_confuse": 0,
        "N_halluc" : 0,
        "N_missing": 0,
    }
    
    
    ##### Per-Message Accounting #####
    # ASSUMPTION: "BGN: ..." / "END: ..." MESSAGES ALWAYS APPEAR IN THE CORRECT ORDER! 
    for datum in data:
        try:
            dtmMsg = datum['msg']
            dtmT   = datum['t']
            dtmDat = datum['data']
        except KeyError:
            print( f'Bad keys!' )
            continue
    
        print( f"\n{TColor.OKBLUE}MESSAGE: {dtmMsg}{TColor.ENDC}\n" ) 
    
        ##### General State ###################################
    
        if "RobotState" in dtmMsg:
            pose_t = np.array( dtmDat['pose'] )
            posn_t = posn_from_xform( pose_t )
            print( f"Moved to {posn_t}" )
            # if np.linalg.norm( posn_t - _SAFE_POSN ) > env_var("_ACCEPT_POSN_ERR"):
            #     shotPoses.append( pose_t )  
            shotPoses.append( pose_t )  
    
        
        ##### Phase 1: Perception #############################
        
        if "BGN: Phase 1" in dtmMsg:
            # ASSUMPTION: PHASE 1 MESSAGE SENT ONLY ONCE PER STEP, See `p1pp2`
            Nstep += 1
            if tStepBgn > 0:
                tStepEnd = dtmT
                print( f"Last iteration took {tStepEnd-tStepBgn:.2f} seconds!" )
            tStepBgn   = dtmT
            tSearchBgn = dtmT
            print( f"\n{TColor.OKGREEN}########## Iteration {Nstep} ######################################################{TColor.ENDC}" )
            # WARNING: THIS SMELLS
            if pNewStep and (not pTryPlan):
                NfailFind += 1
                # symHst.ingest_empty()
            pNewStep   = True
            pTryPlan   = False
            actionFail = False
            ocvTracker.new_scene()
    
        if "ObsMeta"  in dtmMsg:
            inpt = dtmDat['input']
            if _VERBOSE:
                print( f"There are {len( list( inpt.keys() ) )} images at this pose!" )
                imag = inpt[ choice( list( inpt.keys() ) ) ]['image']
                if not _RAM_SAVER:
                    jps.arr_show( imag )
            ocvTracker.process_observation_data( dtmDat, _BLOCK_NAMES, camPose_from_effPose( shotPoses[-1] ) )
    
        if "memory" in dtmMsg:
            if _VERBOSE:
                print( "Post-Search Beliefs" )
                if not _RAM_SAVER:
                    render_memory_list( dtmDat["beliefs"], robotPose = shotPoses )
        
        if "END: Phase 1" in dtmMsg:
            tSearchEnd = dtmT
            if _VERBOSE:
                print( f"Observation took {tSearchEnd-tSearchBgn:.2f} seconds!" )
                print( f"\n{len(dtmDat)} Pre-Cheat Symbols" )
                print( dtmDat )
    
        
        ##### Phase 2: Grounding ##############################
        
        if "END: Phase 2" in dtmMsg:
            
            if _CHK_OUT:
                print( f"{len(dtmDat)} Post-Cheat Symbols" )
                print( dtmDat )
                ocvTracker.log_sensed( dtmDat )
                if not _RAM_SAVER:
                    render_memory_list( syms = dtmDat )
            
            # res = ocvTracker.process_ray_obs()
            res       = ocvTracker.reconcile_scene()
            shotPoses = list()
            if _CHK_OUT:
                print( f"{len(res)} \"Ground Truth\" Symbols" )
                print( res )
                if not _RAM_SAVER:
                    render_memory_list( syms = res )
    
            resCnf = ocvTracker.current_scene_confusion( dtmDat )
            print( f"Scene Confusion: {resCnf}" )
            totlConf = sum_dicts( totlConf, resCnf )
            
            symbols_t = dtmDat[:]
            # symHst.ingest_frame( dtmDat[:] )
            if len( symbols_t ):
                pass
                # Nconf, Nfram, conflicts = symHst.last_frame_confusion( actionSuccess = (not actionFail) )
                # totFound   += Nfram
                # totConfuse += Nconf
    
    
        ##### Phase 3: Planning ###############################
    
        if "BGN: Phase 3" in dtmMsg:
            Nplan += 1
            tPlanBgn = dtmT
            if pNewStep:
                pNewStep = False
                pTryPlan = True
    
        if "Planning Failure" in dtmMsg:
            print( f"{TColor.FAIL}>>>>> PLANNING FAILED <<<<<{TColor.ENDC}" )
            NfailPlan += 1
    
        if "END: Phase 3" in dtmMsg:
            tPlanEnd = dtmT
            tPlanDqu.append( tPlanEnd - tPlanBgn )
            if len( dtmDat ):
                pass
                # symHst.ingest_plan( dtmDat )
    
    
        ##### Phase 4: Execution ##############################
        if "BGN: Phase 4" in dtmMsg:
            Naction   += 1
            actionFail = False
    
        if ("BT END" in dtmMsg):
            if ("fail" in f"{dtmMsg}".lower()):
                print( f"{TColor.FAIL}>>>>> ACTION FAILED <<<<<{TColor.ENDC}" )
                NfailActn += 1
                actionFail = True
    
    print( f"Total Confusion Metrics: {totlConf}" )
    
    ocvTracker.dump_episode( pklFile, nameSimilar = True )

    data = None

    clear_output( wait = False )